# Model Training and Evaluation – Loss, Train/Test, Cross-Validation, Bias–Variance

**Dataset:** Bike counts in Zurich 2024–2025: daily number of bikes at 16 automatic counting stations of the city of
Zurich, with weather and calendar information (Open Data Stadt Zürich). File: `../00_Data/zurich_bike_counts_2024_2025.csv`.

**Business question:** The city wants to estimate the number of cyclists on a working day from the time of year. How
complex should such a model be so that it also works on **new** days?

**After this exercise you can ...**
- compute and compare squared and absolute loss by hand,
- explain why the training error is optimistically biased,
- use a hold-out split and k-fold cross-validation to choose model complexity,
- recognise under- and overfitting and relate them to bias and variance.

## Concept

The concepts behind this exercise (formulas, worked examples and additional explanations) are
explained in the notebook [model_training_evaluation_concept.ipynb](model_training_evaluation_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Loss functions by hand

Two models predict the number of bikes at a counting station on five days:

| day | observed $y$ | model A $\hat y$ | model B $\hat y$ |
|---|---|---|---|
| 1 | 1200 | 1500 | 1250 |
| 2 | 2500 | 2300 | 2550 |
| 3 | 3100 | 3300 | 3050 |
| 4 | 4000 | 3700 | 4050 |
| 5 | 4600 | 4500 | 3800 |

a) Compute the residuals, the **MSE**, the **RMSE** and the **MAE** for both models **by hand** (paper or calculator).

b) Which model is better according to RMSE, and which according to MAE? Explain the difference.

c) Verify your results with `numpy`.

In [ ]:
# Exercise 1: residuals, MSE, RMSE and MAE of two models
y = np.array([1200, 2500, 3100, 4000, 4600])
y_hat_a = np.array([1500, 2300, 3300, 3700, 4500])
y_hat_b = np.array([1250, 2550, 3050, 4050, 3800])

# Compute residuals, MSE, RMSE and MAE for both models
# Your code here

**Your answer:** ...

## Exercise 2: Import and explore the bike counts

a) Import the bike counts. Keep only **working days** (`weekend == 0` and `holiday == 0`) – otherwise the large
difference between working days and weekends dominates everything. Create the column `day_of_year`
(1 = 1 January, …, 366 = 31 December) from the column `date`.

b) Create a scatter plot of `bikes_total` (sum of all 16 counting stations) against `day_of_year`. Is the relationship
linear?

In [ ]:
# a) Import the data as bikes and print its shape:
#    '../00_Data/zurich_bike_counts_2024_2025.csv'
#    Keep only working days, create the column day_of_year
#    (hint: pd.to_datetime(...).dt.dayofyear) and store the result as df.
# Your code here

# b) Scatter plot bikes_total vs. day_of_year
# Your code here

**Your answer:** ...

## Exercise 3: Hold-out split and polynomial models

We describe the number of bikes as a polynomial of the day of the year

$$\hat y = \beta_0 + \beta_1 x + \beta_2 x^2 + \dots + \beta_d x^d .$$

This is still a *linear* regression (linear in the parameters $\beta$). The **degree $d$** controls the model complexity.
The helper function `make_poly_model(degree)` below builds such a model (the features are standardised for numerical
stability).

a) Split the data into a training set (80 %) and a test set (20 %) with `random_state=42`.

b) Fit a polynomial of degree 1 (= simple linear regression) on the training set and compute the RMSE on the training
and on the test set.

In [ ]:
# Function: builds a model with x, x², ..., x^degree and a linear regression
def make_poly_model(degree):
    """Polynomial regression model of the given degree."""
    return make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        StandardScaler(),
        LinearRegression(),
    )

In [ ]:
# Exercise 3: feature and target
X = df[['day_of_year']]
y = df['bikes_total']

# a) Train/test split (80/20, random_state=42)
X_train, X_test, y_train, y_test = ___

# b) Fit a polynomial of degree 1 and compute train and test RMSE
# Your code here

## Exercise 4: Under- and overfitting with a small training sample

Overfitting is most visible when there are only few training examples. Imagine the city had counted on only **30 days**.

a) Draw a random sample of 30 days from the training set (`X_train.sample(30, random_state=13)`, same rows for
`y_train`).

b) For each degree $d = 1, 2, \dots, 15$ fit `make_poly_model(d)` on the 30 days and compute the RMSE on these 30
training days and on the (full) test set. Plot both RMSE curves against the degree (use a logarithmic y-axis).

c) Plot the fitted curves for the degrees 1, 3 and 12 together with the 30 training days.

d) Which degree underfits, which overfits, which fits well? Why does the training error keep decreasing?

In [ ]:
# a) Sample of 30 training days
X_small = X_train.sample(30, random_state=13)
y_small = y_train.loc[X_small.index]

# b) Train and test RMSE for degree 1..15
degrees = range(1, 16)
rmse_train, rmse_test = [], []
for d in degrees:
    # Your code here
    pass

# Table and plot of the train and test RMSE (use plt.yscale('log'))
# Your code here

In [ ]:
# c) Days from the first to the last training day (for smooth curves)
days = np.linspace(
    X_small['day_of_year'].min(), X_small['day_of_year'].max(), 300
)
x_grid = pd.DataFrame({'day_of_year': days})

# Figure showing the 30 training days and the fitted curves
plt.scatter(X_small['day_of_year'], y_small, color='black', label='30 days')
for d in [1, 3, 12]:
    # Fit the model and plot its predictions for the days in x_grid
    # Your code here
    pass
plt.ylim(0, 70000)
plt.xlabel('Day of the year')
plt.ylabel('Bikes per day')
plt.legend()
plt.show()

**Your answer:** ...

## Exercise 5: Choose the degree with 5-fold cross-validation

In practice we must not look at the test set to choose the degree. Instead we use k-fold cross-validation **on the
training data only**.

a) For $d = 1, \dots, 10$ compute the 5-fold CV RMSE on the **30 training days**
(use `cross_val_score(..., scoring='neg_root_mean_squared_error', cv=KFold(5, shuffle=True, random_state=42))`;
note the minus sign). Which degree does CV select?

b) Repeat a) with the **full training set**. Which degree is selected now and how large is the difference between
degree 3 and degree 10?

c) Fit the selected model on the full training set and evaluate it **once** on the test set.

d) Why does overfitting become much less severe with more data?

In [ ]:
# Exercise 5: 5-fold cross-validation for polynomial degrees
cv = KFold(n_splits=5, shuffle=True, random_state=42)


# Function (to complete): returns the mean 5-fold CV RMSE for one degree
def cv_rmse(degree, X, y):
    """Mean 5-fold CV RMSE of a polynomial model."""
    # Your code here
    pass


# a) and b) CV RMSE by degree for 30 days and for the full training set
degrees_cv = list(range(1, 11))
cv_small, cv_full = [], []
# Your code here

In [ ]:
# c) Fit the selected model on the full training set,
#    evaluate once on the test set
# Your code here

**Your answer:** ...

## Exercise 6: Visualise bias and variance

Bias and variance describe what happens if we **repeat** the whole training on *other* samples of the same size.

a) Draw 50 different random samples of 30 days from the training set (`random_state=0, 1, ..., 49`). For each sample
fit a polynomial of degree 1, 3 and 12. Plot the 50 fitted curves per degree in three subplots
(use `ylim(0, 70000)`).

b) For each degree, compute the mean and the standard deviation of the 50 predictions for **day 180** (end of June).
What do you observe? Relate your observations to bias and variance.

In [ ]:
# Exercise 6: days of the year and day 180 (for the predictions)
x_grid = pd.DataFrame({'day_of_year': np.linspace(1, 366, 300)})
x_180 = pd.DataFrame({'day_of_year': [180]})
pred_180 = {1: [], 3: [], 12: []}

# a) Figure with one plot per degree: 50 fits on random samples of 30 days
plt.figure(figsize=(15, 4))
for i, d in enumerate([1, 3, 12]):
    plt.subplot(1, 3, i + 1)
    for seed in range(50):
        # Draw a sample of 30 training days (random_state=seed), fit the
        # model, plot its curve and append the prediction for day 180 to
        # pred_180[d]
        # Your code here
        pass
    plt.ylim(0, 70000)
    plt.title('degree ' + str(d))
    plt.xlabel('Day of the year')
plt.show()

# b) Mean and standard deviation of the predictions for day 180
# Your code here

**Your answer:** ...

## Exercise 7: More covariates instead of a more complex curve

Exercise 5 showed that a higher polynomial degree does not reduce the error much below about 6'400 bikes per day. The
remaining error is "noise" only from the point of view of a model that knows nothing but the date. Weather and holidays
explain part of it.

a) Fit a linear regression with a polynomial of degree 3 in `day_of_year` **plus** the covariates `temp_mean`,
`rain_duration_min`, `solar_radiation` and `school_holiday` (the code for the features is given).

b) Compare its 5-fold CV RMSE and test RMSE with the polynomial of degree 3 without covariates.

c) What does this tell you about bias, variance and the noise term?

In [ ]:
# Exercise 7: date features plus weather and holiday covariates
covariates = [
    'temp_mean',
    'rain_duration_min',
    'solar_radiation',
    'school_holiday',
]


# Function: creates day_of_year, its square and cube plus the covariates
def date_and_weather_features(rows):
    """Polynomial of degree 3 in day_of_year plus weather and holidays."""
    X = df.loc[rows, ['day_of_year'] + covariates].copy()
    X['day_of_year_2'] = X['day_of_year'] ** 2
    X['day_of_year_3'] = X['day_of_year'] ** 3
    return X


# Features of the training and test set
Xc_train = date_and_weather_features(X_train.index)
Xc_test = date_and_weather_features(X_test.index)

# The two models: date only, and date plus covariates
model_date = make_poly_model(3)
model_cov = make_pipeline(StandardScaler(), LinearRegression())

# a) and b) 5-fold CV RMSE and test RMSE of both models
# Your code here

**Your answer:** ...

## Exercise 8: Reflection

a) A colleague chooses the polynomial degree by trying all degrees and picking the one with the smallest **test** error.
What is the problem?

b) Why is k-fold cross-validation usually preferred over a single validation split when the data set is small?

c) Name two ways to reduce the variance of a model and one way to reduce its bias.

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')